# Second case: the tier's question and Marketing's pushback, argued from the same numbers

**Week 1, Tuesday afternoon. The second case, in pairs, forty minutes.** One of you answers Marketing,
the other answers the head of Retail-Plus, from the same file, and together you write the reply.

> **Marketing comes back with a new deck.** "Student orders per customer rose 40 percent after our
> campus push, so acquisition works. And Retail-Plus web orders fell hardest, 24 to 9, so this is the
> website team's problem, not the tier's and not the app's."
>
> The marketing lead, Kalpa Retail

> **The tier asks.** "Fine. Which of my members do I call first?"
>
> The head of Retail-Plus

> **Kavya's review of the escalated case.** "You found a branch that moved for a reason nobody
> guessed. Do the same with Marketing's two new numbers: find what each is made of before you agree
> or disagree."

This is the exercise twin: each placeholder is a lettered choice in the comment above it. Run it from the top; it stops at the first placeholder you have not filled, which is intended.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit
ORDERS = kit.load_records("C2_W01_D02_orders_STUDENT.py")
SEGMENTS = ["Retail-Core", "Retail-Plus", "Business", "Student"]


def tree_for(rows):
    """The revenue tree's leaves for any list of orders, returned as a dictionary (chapter 3)."""
    revenue = 0
    seen = {}
    for order in rows:
        revenue += order["amount"]
        seen[order["customer_id"]] = True
    orders = len(rows)
    customers = len(seen)
    return {"revenue": revenue, "orders": orders, "customers": customers,
            "orders_per_customer": orders / customers if customers else 0,
            "revenue_per_order": revenue / orders if orders else 0}


def pct_change(before, after):
    """Chapter 5's fixed helper: the change every time, in the same type."""
    return round(100 * (after - before) / before, 1)

print(len(ORDERS), "orders loaded")

In [ ]:
kit.side_by_side(
    kit.ladder(["Student +40 percent: acquisition?", "Web fell hardest: the website?",
                "Who to call first", "The reply and the evidence"], show=False),
    kit.matrix(["Marketing", "the head of Retail-Plus"], ["claim", "test"],
               [["campaigns work; the website", "new ids; web in another segment"], ["which members", "orders per member, Q1 to Q2"]],
               show=False),
)

## Part 1. "Student orders per customer rose 40 percent, so acquisition works"

Acquisition means new customers. Count the Student ids in each quarter and the ones that are new.

In [ ]:
s_ids = {q: {o["customer_id"] for o in ORDERS if o["segment"] == "Student" and o["quarter"] == q} for q in ("Q1", "Q2")}
s_orders = {q: sum(1 for o in ORDERS if o["segment"] == "Student" and o["quarter"] == q) for q in ("Q1", "Q2")}
# TODO 1. Which set holds the Student customers who are new in Q2?
#   a) s_ids["Q2"] - s_ids["Q1"]
#   b) s_ids["Q1"] | s_ids["Q2"]
#   c) s_ids["Q1"] & s_ids["Q2"]
#   d) len(s_ids["Q2"]) - len(s_ids["Q1"])
new_students = __TODO1__
kit.columns(["Q1", "Q2"], [("Student customers", [len(s_ids["Q1"]), len(s_ids["Q2"])]), ("Student orders", [s_orders["Q1"], s_orders["Q2"]])],
            width=520, title="Student customers and orders, Q1 and Q2")
kit.check("your set matches a second route over the Q2 ids", new_students == {c for c in s_ids["Q2"] if c not in s_ids["Q1"]})
kit.check("the Student order counts add up to the file", s_orders["Q1"] + s_orders["Q2"] == sum(1 for o in ORDERS if o["segment"] == "Student"))

**Item 1 in your brief** goes in now, from what this part printed.

## Part 2. "Retail-Plus web orders fell hardest, so it is the website"

A broken website hurts every customer who uses it. If the website were the cause, what would another
segment's web orders show?

In [ ]:
web = {(o["segment"], o["quarter"]): 0 for o in ORDERS}
for o in ORDERS:
    if o["channel"] == "web":
        web[(o["segment"], o["quarter"])] += 1
# TODO 2. If the website were broken for everyone, which other segment's web orders would also have fallen hard?
#   a) "Retail-Plus"
#   b) "Student"
#   c) "Retail-Core"
#   d) "none"
also_falls = __TODO2__
picked = also_falls if also_falls in SEGMENTS else "Retail-Plus"
kit.columns(["Retail-Plus", picked], [("Q1 web orders", [web[("Retail-Plus", "Q1")], web[(picked, "Q1")]]),
                                      ("Q2 web orders", [web[("Retail-Plus", "Q2")], web[(picked, "Q2")]])],
            width=560, title="Web orders: the members against the segment you chose")
kit.check("your pick is a segment that can test the website claim",
          also_falls in SEGMENTS and also_falls != "Retail-Plus" and web[(also_falls, "Q1")] >= 10)
kit.check("Retail-Plus web orders fell 24 to 9", (web[("Retail-Plus", "Q1")], web[("Retail-Plus", "Q2")]) == (24, 9))

**Item 2 in your brief** goes in now, from what this part printed.

## Part 3. "Which of my members do I call first?"

Count each member's orders in Q1 and Q2, and decide who goes first.

In [ ]:
per = {q: {} for q in ("Q1", "Q2")}
for o in ORDERS:
    if o["segment"] == "Retail-Plus":
        per[o["quarter"]][o["customer_id"]] = per[o["quarter"]].get(o["customer_id"], 0) + 1
pairs = {}
for cid in per["Q1"]:
    key = (per["Q1"][cid], per["Q2"].get(cid, 0))
    pairs[key] = pairs.get(key, 0) + 1
call_first = []
for cid in per["Q1"]:
    q1n, q2n = per["Q1"][cid], per["Q2"].get(cid, 0)
    # TODO 3. Which members go to the top of the call list?
    #   a) q2n == 0
    #   b) q1n - q2n >= 2
    #   c) q2n > q1n
    #   d) q1n == 1
    if __TODO3__:
        call_first.append(cid)
labels = [f"{a} to {b}" for a, b in sorted(pairs, reverse=True)]
kit.bars([(l, pairs[k]) for l, k in zip(labels, sorted(pairs, reverse=True))],
         title="Retail-Plus members, by orders in Q1 and in Q2")
drops = {c: per["Q1"][c] - per["Q2"].get(c, 0) for c in per["Q1"]}
kit.check("your list matches a second route over the per-member counts", set(call_first) == {c for c in drops if drops[c] == max(drops.values())})
kit.check("the per-member counts add back to the tier's orders", sum(per["Q1"].values()) + sum(per["Q2"].values()) == sum(1 for o in ORDERS if o["segment"] == "Retail-Plus"))

**Item 3 in your brief** goes in now, from what this part printed.

## Part 4. The reply, and the one request that tests the most

Item 4 in your brief goes in once this part has run.

In [ ]:
EVIDENCE = {
    "campaigns": "Marketing's campaign reach by month for the July student push",
    "export": "this export again, cut by city, channel and week from July",
    "app_logs": "the app's reorder logs by week since the 25 August release",
    "tier_log": "the tier's July change log, renewals and support tickets",
}
# TODO 4. The fall began in July and the button broke on 25 August. Which request tests the cause behind the larger part of the fall?
#   a) "campaigns"
#   b) "export"
#   c) "app_logs"
#   d) "tier_log"
first_request = __TODO4__
kit.tree({"label": "the reply", "kind": "lit", "branches": [
    ("to Marketing", {"label": f"Student: {len(new_students)} new ids\nweb tested on {also_falls}", "kind": "known"}),
    ("to the tier", {"label": f"call the {len(call_first)}\nat the top of the list", "kind": "known"}),
    ("first request", {"label": EVIDENCE[first_request][:34], "kind": "unknown"})]},
    title="The pair's reply, in three branches")
import hashlib
kit.check("your pick matches the key, stored as a fingerprint", hashlib.sha256(first_request.encode()).hexdigest()[:10] == "a49850fe17")
kit.check("your pick names one of the four sources", first_request in EVIDENCE)

In [ ]:
kit.check_summary()